________________________________________________________________________________________________________________________________________________________

In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import pickle
from deepface import DeepFace
from ultralytics import YOLO
from sklearn.metrics.pairwise import cosine_similarity
from albumentations import (
    HorizontalFlip, RandomBrightnessContrast, Rotate, ShiftScaleRotate, Blur, Compose
)
from albumentations.augmentations.transforms import GaussNoise

## Dataset Augmentation

In [ ]:
import os
import cv2
import numpy as np
# Define augmentation pipeline
augmentation_pipeline = Compose([
    HorizontalFlip(p=0.5),  # Flip the image horizontally with 50% probability
    Rotate(limit=15, p=0.7),  # Rotate within -15 to +15 degrees
    ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.7),
    RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    Blur(blur_limit=3, p=0.3),  # Apply blur
    GaussNoise(var_limit=(10.0, 50.0), p=0.3),  # Add noise
])

# Function to apply augmentation
def augment_image(image, num_augmentations=15):
    augmented_images = []
    for _ in range(num_augmentations):
        augmented = augmentation_pipeline(image=image)
        augmented_images.append(augmented["image"])
    return augmented_images

# Path to dataset
input_folder = "D:\\Datasets\\Facial Recognition"
output_folder = "D:\\Datasets\\Facial Recognition Augmented"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Augment images for each person
for person_folder in os.listdir(input_folder):
    person_path = os.path.join(input_folder, person_folder)
    output_person_path = os.path.join(output_folder, person_folder)

    if not os.path.exists(output_person_path):
        os.makedirs(output_person_path)

    for img_name in os.listdir(person_path):
        img_path = os.path.join(person_path, img_name)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        augmented_images = augment_image(img)

        # Save original and augmented images
        cv2.imwrite(os.path.join(output_person_path, img_name), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
        for i, aug_img in enumerate(augmented_images):
            aug_img_path = os.path.join(output_person_path, f"{os.path.splitext(img_name)[0]}_aug_{i}.jpg")
            cv2.imwrite(aug_img_path, cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))


## Getting face embeddings and storing it in a pickle file

In [3]:
def get_face_embeddings(image_folder, metadata_file):
    embeddings = []
    metadata = pd.read_csv(metadata_file)

    for idx, row in metadata.iterrows():
        folder_path = os.path.join(image_folder, row["folder"])
        for filename in os.listdir(folder_path):
            img_path = os.path.join(folder_path, filename)
            try:
                embedding = DeepFace.represent(img_path, model_name="Facenet", enforce_detection=False)
                if isinstance(embedding, list):
                    # Append embedding with metadata
                    embeddings.append((embedding[0]["embedding"], row["name"], row["cnic"], row["age"]))
            except Exception as e:
                print(f"Error processing {img_path}: {e}")
                continue
    return embeddings

In [4]:
def is_criminal(new_embedding, criminal_embeddings, threshold=0.6):
    # Reshape new_embedding to 2D
    new_embedding = np.array(new_embedding).reshape(1, -1)

    similarities = []
    for e in criminal_embeddings:
        emb = np.array(e[0]).reshape(1, -1)  # Ensure criminal embeddings are 2D
        similarities.append(cosine_similarity(new_embedding, emb)[0, 0])

    max_similarity = max(similarities)

    if max_similarity > threshold:
        index = np.argmax(similarities)
        _, name, cnic, age = criminal_embeddings[index]
        return True, max_similarity, name, cnic, age

    return False, max_similarity, None, None, None

In [4]:
# Generate embeddings with metadata
image_folder = "D:\\Datasets\\Facial Recognition Augmented\\"
metadata_file = "D:\\Datasets\\Facial Recognition Augmented\\metadata.csv"
criminal_embeddings = get_face_embeddings(image_folder, metadata_file)

In [21]:
# Save embeddings to a file using pickle
with open("D:\\Datasets\\Facial Recognition Augmented\\embeddings.pkl", "wb") as f:
    pickle.dump(criminal_embeddings, f)

## Method 1: haarcascades

In [40]:
# Open the video file
video_path = "D:\\Datasets\\Facial Recognition\\test_video.mp4"
output_path = "D:\\Datasets\\Facial Recognition\\output_video.mp4"
cap = cv2.VideoCapture(video_path)

# Define the codec and create VideoWriter object
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for .mp4 format
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Convert frame to grayscale for face detection
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    face_detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    faces = face_detector.detectMultiScale(gray_frame, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    for (x, y, w, h) in faces:
        face_img = frame[y:y+h, x:x+w]

        # Get embedding for the detected face
        new_embedding = get_embedding_for_face(face_img)

        if new_embedding is not None:
            # Check if the face matches a criminal embedding
            criminal_detected, similarity, name, cnic, age = is_criminal(new_embedding, criminal_embeddings, threshold=0.5)

            if criminal_detected:
                # Draw bounding box and metadata
                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)  # Red bounding box
                cv2.putText(frame, f"{name}, {age} yrs", (x, y - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                cv2.putText(frame, f"CNIC: {cnic}", (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

    # Write the frame with detections to the output video
    out.write(frame)

    # Display the frame for debugging
    cv2.imshow("Video", frame)

    # Break the loop with 'q' key
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# Release resources
cap.release()
out.release()  # Save the video
cv2.destroyAllWindows()

print(f"Processed video saved to {output_path}")

Processed video saved to D:\Datasets\Facial Recognition\output_video.mp4


## Method 2: YOLO

In [ ]:
with open("D:\\Datasets\\Facial Recognition Augmented\\embeddings.pkl", "rb") as f:
    criminal_embeddings = pickle.load(f)
    
model = YOLO("D:\\Datasets\\Facial Recognition Augmented\\yolov8n-face.pt")  # Replace with your YOLO model file path
# Open the video file
video_path = "D:\\Datasets\\test\\test_video_2.mp4"
output_path = "D:\\Datasets\\Outputs\\output_video_2.mp4"
cap = cv2.VideoCapture(video_path)

# Define the codec and create a VideoWriter object
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

# Process video frame by frame
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO detection
    results = model(frame)
    detections = results[0].boxes.xyxy.cpu().numpy()  # Bounding boxes
    confidences = results[0].boxes.conf.cpu().numpy() if hasattr(results[0].boxes, 'conf') else [1.0] * len(detections)

    for detection, confidence in zip(detections, confidences):
        x1, y1, x2, y2 = map(int, detection[:4])  # Bounding box coordinates
        if confidence < 0.57:  # Confidence threshold
            continue

        # Extract face ROI
        face_img = frame[y1:y2, x1:x2]

        # Get embedding for the detected face
        try:
            new_embedding = DeepFace.represent(face_img, model_name="Facenet", enforce_detection=False)
            if isinstance(new_embedding, list):
                new_embedding = new_embedding[0]["embedding"]
        except Exception as e:
            print(f"Error processing face: {e}")
            continue

        if new_embedding is not None:
            # Check if the face matches a criminal embedding
            criminal_detected, similarity, name, cnic, age = is_criminal(new_embedding, criminal_embeddings, threshold=0.58)

            if criminal_detected:
                # Draw bounding box and metadata
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)  # Red bounding box
                cv2.putText(frame, f"{name}, {age} yrs", (x1, y1 - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                cv2.putText(frame, f"CNIC: {cnic}", (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

    # Write the processed frame to the output video
    out.write(frame)

    # Display the frame (optional for debugging)
    #cv2.imshow("Video", frame)
    #if cv2.waitKey(1) & 0xFF == ord("q"):
    #    break

# Release resources
cap.release()
out.release()
#cv2.destroyAllWindows()

print(f"Processed video saved to {output_path}")



0: 384x640 (no detections), 9.7ms
Speed: 2.3ms preprocess, 9.7ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 9.0ms
Speed: 1.7ms preprocess, 9.0ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 9.0ms
Speed: 1.8ms preprocess, 9.0ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 9.4ms
Speed: 1.6ms preprocess, 9.4ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 9.3ms
Speed: 1.8ms preprocess, 9.3ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 8.4ms
Speed: 1.8ms preprocess, 8.4ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 8.5ms
Speed: 1.9ms preprocess, 8.5ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 8.6ms
Speed: 2.0ms preprocess, 8.6ms inference, 0.8ms 

# Adding Timestamp Logic

In [ ]:
import cv2
import pickle
from deepface import DeepFace

with open("D:\\Datasets\\Facial Recognition Augmented\\embeddings.pkl", "rb") as f:
    criminal_embeddings = pickle.load(f)
# Load YOLO model
model = YOLO("D:\\Datasets\\Models\\yolov8n-face.pt")  # Replace with your YOLO model file path

# Open the video file
video_path = "D:\\Datasets\\test\\test_video_2.mp4"
output_path = "D:\\Datasets\\Outputs\\output_video_2.mp4"
timestamps_path = "D:\\Datasets\\Outputs\\timestamps.txt"
cap = cv2.VideoCapture(video_path)

# Define the codec and create a VideoWriter object
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

# Open file to save timestamps
timestamps_file = open(timestamps_path, "w")

# Process video frame by frame
frame_counter = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO detection
    results = model(frame)
    detections = results[0].boxes.xyxy.cpu().numpy()  # Bounding boxes
    confidences = results[0].boxes.conf.cpu().numpy() if hasattr(results[0].boxes, 'conf') else [1.0] * len(detections)

    for detection, confidence in zip(detections, confidences):
        x1, y1, x2, y2 = map(int, detection[:4])  # Bounding box coordinates
        if confidence < 0.57:  # Confidence threshold
            continue

        # Extract face ROI
        face_img = frame[y1:y2, x1:x2]

        # Get embedding for the detected face
        try:
            new_embedding = DeepFace.represent(face_img, model_name="Facenet", enforce_detection=False)
            if isinstance(new_embedding, list):
                new_embedding = new_embedding[0]["embedding"]
        except Exception as e:
            print(f"Error processing face: {e}")
            continue

        if new_embedding is not None:
            # Check if the face matches a criminal embedding
            criminal_detected, similarity, name, cnic, age = is_criminal(new_embedding, criminal_embeddings, threshold=0.58)

            if criminal_detected:
                # Calculate timestamp in seconds
                timestamp = frame_counter / fps

                # Save timestamp to file
                timestamps_file.write(f"{timestamp:.2f}\n")

                # Draw bounding box and metadata
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)  # Red bounding box
                cv2.putText(frame, f"{name}, {age} yrs", (x1, y1 - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                cv2.putText(frame, f"CNIC: {cnic}", (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

    # Write the processed frame to the output video
    out.write(frame)

    # Increment frame counter
    frame_counter += 1

    # Display the frame (optional for debugging)
    # cv2.imshow("Video", frame)
    # if cv2.waitKey(1) & 0xFF == ord("q"):
    #     break

# Release resources
cap.release()
out.release()
timestamps_file.close()
cv2.destroyAllWindows()
print(f"Processed video saved to {output_path}")
print(f"Timestamps saved to {timestamps_path}")
